In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# Real data
tickers = ["AAPL", "MSFT"]
data = yf.download(tickers, start="2020-01-01", end="2024-01-01")["Adj Close"]
returns = data.pct_change().dropna()

# Weights
weights = np.array([0.6, 0.4])

# Step 2: Portfolio weights
weights = np.array([0.6, 0.4])

# Step 3: Monte Carlo simulation (FULL PATHS — this is key)
num_simulations = 100
horizon = 252

portfolio_paths = []

for _ in range(num_simulations):
    sampled_returns = returns.sample(n=horizon, replace=True).values
    portfolio_returns = sampled_returns @ weights
    portfolio_values = np.cumprod(1 + portfolio_returns)
    portfolio_paths.append(portfolio_values)

portfolio_paths = np.array(portfolio_paths)

# Step 4: Plot
plt.figure(figsize=(10,6))

for path in portfolio_paths[:20]:
    plt.plot(path, color='gray', alpha=0.3)

plt.title("Monte Carlo Simulation of Portfolio Paths")
plt.xlabel("Days")
plt.ylabel("Portfolio Value")

# 🔥 THIS IS WHAT YOU NEED
plt.savefig("simulated_paths.png", bbox_inches="tight")

plt.show()


/tmp/ipykernel_703/163413918.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start="2020-01-01", end="2024-01-01")["Adj Close"]
[*********************100%***********************]  2 of 2 completed


KeyError: 'Adj Close'

In [ ]:
final_values = portfolio_paths[:, -1]

def calculate_var(values, alpha=0.05):
    return np.percentile(values, alpha * 100)

def calculate_cvar(values, alpha=0.05):
    var = calculate_var(values, alpha)
    return values[values <= var].mean()

var_95 = calculate_var(final_values)
cvar_95 = calculate_cvar(final_values)

print("VaR (5%):", var_95)
print("CVaR (5%):", cvar_95)

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(final_values, bins=30, alpha=0.7)

plt.axvline(var_95, linestyle="--", label="VaR 5%")
plt.axvline(cvar_95, linestyle="--", label="CVaR 5%")

plt.title("Distribution of Final Portfolio Values")
plt.xlabel("Final Value")
plt.ylabel("Frequency")

plt.legend()
plt.grid(True)

plt.savefig("var_cvar_chart.png", bbox_inches="tight")

plt.show()

In [ ]:
def compute_drawdown(path):
    running_max = np.maximum.accumulate(path)
    drawdown = (path - running_max) / running_max
    return drawdown

In [ ]:
plt.figure(figsize=(10, 6))

for path in portfolio_paths[:5]:
    dd = compute_drawdown(path)
    plt.plot(dd, alpha=0.7)

plt.title("Drawdown Over Time")
plt.xlabel("Days")
plt.ylabel("Drawdown")

plt.grid(True)

plt.savefig("drawdown_chart.png", bbox_inches="tight")

plt.show()